# Prescriptive Analytics — Inventory Optimization

This notebook is a **separate prescriptive-analytics implementation** for the Historical Product Demand dataset. It includes data preparation, demand aggregation, Linear Programming (LP), Integer Programming (IP), Mixed-Integer Programming (MIP), sensitivity analysis, evaluation, and visualizations.

**Important:** The dataset contains historical demand, not current stock, supplier lead times, or actual costs. The cost values and capacity below are editable assumptions for demonstration. Results are an optimization scenario, not a real purchase order recommendation.

## 1. Install dependencies
Run this cell once in Jupyter. If your environment already has the packages, you can skip it.

In [ ]:
%pip install pandas numpy scipy pulp matplotlib

## 2. Import libraries and configure inputs

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog
import pulp

# Update this path to the location of your CSV file.
DATA_PATH = Path("Historical Product Demand.csv")

# Editable scenario assumptions (not provided by the dataset).
HOLDING_COST_PER_UNIT = 2.5
ORDERING_COST_PER_PRODUCT = 50.0
MAX_TOTAL_CAPACITY = 5000
DEMAND_INCREASE_PCT = 10
TOP_N_PRODUCTS_TO_PLOT = 20

## 3. Load and inspect the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
print("\\nData types:")
print(df.dtypes)
print("\\nMissing values:")
print(df.isna().sum())

## 4. Clean and prepare the data
Demand values in this dataset can contain parentheses or commas. Invalid demand/date rows are removed; negative demand is excluded from this inventory scenario.

In [ ]:
required = ["Product_Code", "Warehouse", "Product_Category", "Date", "Order_Demand"]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

df = df.copy()
df["Order_Demand"] = (
    df["Order_Demand"].astype("string")
      .str.replace("(", "", regex=False)
      .str.replace(")", "", regex=False)
      .str.replace(",", "", regex=False)
)
df["Order_Demand"] = pd.to_numeric(df["Order_Demand"], errors="coerce")
df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

df = df.dropna(subset=required).copy()
df = df[df["Order_Demand"] >= 0].copy()
df["Order_Demand"] = df["Order_Demand"].astype(float)

print("Cleaned shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())
display(df.head())

## 5. Aggregate historical demand
The optimization unit is a **Product_Code × Warehouse** pair, so products with different warehouse demand remain separate.

In [ ]:
demand = (
    df.groupby(["Product_Code", "Warehouse"], as_index=False)
      .agg(
          Product_Category=("Product_Category", lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0]),
          Historical_Demand=("Order_Demand", "sum"),
          Demand_Records=("Order_Demand", "size"),
      )
)
demand["Historical_Demand"] = demand["Historical_Demand"].clip(lower=0)
demand = demand.sort_values("Historical_Demand", ascending=False).reset_index(drop=True)

print("Product-warehouse combinations:", len(demand))
display(demand.head(10))

## 6. Define the optimization scenario
This model chooses a quantity to cover the scenario demand. With positive per-unit costs and only a minimum-demand constraint, the mathematical optimum is usually exactly demand. The capacity constraint can make the model infeasible if total required demand exceeds capacity. For realistic replenishment planning, replace historical totals with a planning-horizon forecast and include current stock, lead time, and supplier constraints.

In [ ]:
# For demonstration, scale historical demand to a manageable planning scenario.
# This is an explicit scenario assumption, not a forecast.
planning = demand.copy()
planning["Required_Demand"] = planning["Historical_Demand"]

# If the full historical total exceeds the illustrative capacity, use the top products
# only for the constrained example. Set USE_TOP_PRODUCTS=False to optimize all rows.
USE_TOP_PRODUCTS = True
if USE_TOP_PRODUCTS and planning["Required_Demand"].sum() > MAX_TOTAL_CAPACITY:
    planning = planning.head(50).copy()

planning = planning.reset_index(drop=True)
n = len(planning)
required_demand = planning["Required_Demand"].to_numpy(dtype=float)

print("Products in optimization:", n)
print("Total required demand:", required_demand.sum())
print("Capacity:", MAX_TOTAL_CAPACITY)

## 7. Linear Programming (LP)
Continuous decision variables allow fractional units. The model minimizes assumed procurement/holding cost while meeting required demand and respecting total capacity.

In [ ]:
unit_cost = HOLDING_COST_PER_UNIT
# Use a positive unit cost; fixed ordering cost is modeled separately in the integer section.
c_lp = np.full(n, unit_cost, dtype=float)

# Demand constraints: x_i >= demand_i  ->  -x_i <= -demand_i
A_ub = [-np.ones(n)]
b_ub = [-required_demand]
# Capacity: sum(x_i) <= capacity
A_ub.append(np.ones(n))
b_ub.append(MAX_TOTAL_CAPACITY)
A_ub = np.vstack(A_ub)
b_ub = np.concatenate(b_ub)

lp_result = linprog(
    c=c_lp,
    A_ub=A_ub,
    b_ub=b_ub,
    bounds=[(0, None)] * n,
    method="highs",
)

if lp_result.success:
    planning["LP_Order_Quantity"] = lp_result.x
    print("LP status:", lp_result.message)
    print("LP objective cost:", round(lp_result.fun, 2))
    display(planning[["Product_Code", "Warehouse", "Required_Demand", "LP_Order_Quantity"]].head(10))
else:
    print("LP did not find a feasible solution:", lp_result.message)
    print("Check whether total required demand exceeds the capacity.")

## 8. Integer Programming (IP)
Integer quantities are appropriate when products cannot be ordered fractionally. The fixed ordering cost is included per product-warehouse pair, using a binary order-activation variable.

In [ ]:
ip_model = pulp.LpProblem("Inventory_Integer_Optimization", pulp.LpMinimize)

q_ip = {
    i: pulp.LpVariable(f"qty_{i}", lowBound=0, cat="Integer")
    for i in range(n)
}
order_ip = {
    i: pulp.LpVariable(f"order_on_{i}", cat="Binary")
    for i in range(n)
}

# A safe linking upper bound for each variable.
big_m = max(float(MAX_TOTAL_CAPACITY), float(required_demand.max()) if n else 0.0)

ip_model += pulp.lpSum(
    HOLDING_COST_PER_UNIT * q_ip[i] + ORDERING_COST_PER_PRODUCT * order_ip[i]
    for i in range(n)
)

for i in range(n):
    ip_model += q_ip[i] >= int(np.ceil(required_demand[i])), f"demand_{i}"
    ip_model += q_ip[i] <= big_m * order_ip[i], f"activate_order_{i}"

ip_model += pulp.lpSum(q_ip[i] for i in range(n)) <= MAX_TOTAL_CAPACITY, "capacity"

ip_model.solve(pulp.PULP_CBC_CMD(msg=False))
ip_status = pulp.LpStatus[ip_model.status]
print("IP status:", ip_status)

if ip_status == "Optimal":
    planning["IP_Order_Quantity"] = [pulp.value(q_ip[i]) for i in range(n)]
    print("IP objective cost:", round(pulp.value(ip_model.objective), 2))
    display(planning[["Product_Code", "Warehouse", "Required_Demand", "IP_Order_Quantity"]].head(10))
else:
    print("No optimal IP solution. Review demand, capacity, and assumptions.")

## 9. Mixed-Integer Programming (MIP)
This version uses integer order quantities and binary order-activation decisions. This is a common mixed-integer structure; it does not mix fractional units with integer units for the same product.

In [ ]:
mip_model = pulp.LpProblem("Inventory_Mixed_Integer_Optimization", pulp.LpMinimize)

q_mip = {
    i: pulp.LpVariable(f"mip_qty_{i}", lowBound=0, cat="Integer")
    for i in range(n)
}
use_mip = {
    i: pulp.LpVariable(f"mip_order_on_{i}", cat="Binary")
    for i in range(n)
}

mip_model += pulp.lpSum(
    HOLDING_COST_PER_UNIT * q_mip[i] + ORDERING_COST_PER_PRODUCT * use_mip[i]
    for i in range(n)
)

for i in range(n):
    mip_model += q_mip[i] >= int(np.ceil(required_demand[i])), f"mip_demand_{i}"
    mip_model += q_mip[i] <= big_m * use_mip[i], f"mip_link_{i}"

mip_model += pulp.lpSum(q_mip[i] for i in range(n)) <= MAX_TOTAL_CAPACITY, "mip_capacity"
mip_model.solve(pulp.PULP_CBC_CMD(msg=False))

mip_status = pulp.LpStatus[mip_model.status]
print("MIP status:", mip_status)

if mip_status == "Optimal":
    planning["MIP_Order_Quantity"] = [pulp.value(q_mip[i]) for i in range(n)]
    print("MIP objective cost:", round(pulp.value(mip_model.objective), 2))
    display(planning[["Product_Code", "Warehouse", "Required_Demand", "MIP_Order_Quantity"]].head(10))
else:
    print("No optimal MIP solution. Review demand, capacity, and assumptions.")

## 10. Sensitivity analysis: increase demand by 10%
Re-run the integer model under a higher-demand scenario and compare order quantities. The capacity may become binding or make the scenario infeasible.

In [ ]:
sensitivity = planning.copy()
sensitivity["Scenario_Demand"] = np.ceil(
    sensitivity["Required_Demand"] * (1 + DEMAND_INCREASE_PCT / 100)
).astype(int)

sens_model = pulp.LpProblem("Demand_Sensitivity", pulp.LpMinimize)
q_sens = {
    i: pulp.LpVariable(f"sens_qty_{i}", lowBound=0, cat="Integer")
    for i in range(n)
}
z_sens = {
    i: pulp.LpVariable(f"sens_order_on_{i}", cat="Binary")
    for i in range(n)
}

sens_model += pulp.lpSum(
    HOLDING_COST_PER_UNIT * q_sens[i] + ORDERING_COST_PER_PRODUCT * z_sens[i]
    for i in range(n)
)
for i in range(n):
    sens_model += q_sens[i] >= int(sensitivity.loc[i, "Scenario_Demand"]), f"sens_demand_{i}"
    sens_model += q_sens[i] <= big_m * z_sens[i], f"sens_link_{i}"
sens_model += pulp.lpSum(q_sens[i] for i in range(n)) <= MAX_TOTAL_CAPACITY, "sens_capacity"
sens_model.solve(pulp.PULP_CBC_CMD(msg=False))

sens_status = pulp.LpStatus[sens_model.status]
print("Sensitivity status:", sens_status)
if sens_status == "Optimal":
    sensitivity["Sensitivity_Order_Quantity"] = [pulp.value(q_sens[i]) for i in range(n)]
    sensitivity["Quantity_Change"] = (
        sensitivity["Sensitivity_Order_Quantity"] - sensitivity.get("IP_Order_Quantity", sensitivity["Required_Demand"])
    )
    display(sensitivity[[
        "Product_Code", "Warehouse", "Required_Demand",
        "Scenario_Demand", "Sensitivity_Order_Quantity", "Quantity_Change"
    ]].head(10))
else:
    print("The increased-demand scenario is infeasible under the current capacity.")

## 11. Evaluate the solution
The following is an illustrative cost calculation. Revenue/profit cannot be calculated reliably from this dataset alone because selling price, realized sales, and actual cost data are not supplied.

In [ ]:
if "IP_Order_Quantity" in planning.columns:
    evaluation_qty = planning["IP_Order_Quantity"].astype(float)
elif "LP_Order_Quantity" in planning.columns:
    evaluation_qty = planning["LP_Order_Quantity"].astype(float)
else:
    evaluation_qty = planning["Required_Demand"].astype(float)

evaluation = pd.DataFrame({
    "Metric": [
        "Total required demand",
        "Total optimized quantity",
        "Estimated variable holding/procurement cost",
        "Estimated fixed ordering cost",
        "Estimated total scenario cost",
        "Capacity utilization (%)",
    ],
    "Value": [
        float(planning["Required_Demand"].sum()),
        float(evaluation_qty.sum()),
        float(evaluation_qty.sum() * HOLDING_COST_PER_UNIT),
        float((evaluation_qty > 0).sum() * ORDERING_COST_PER_PRODUCT),
        float(evaluation_qty.sum() * HOLDING_COST_PER_UNIT + (evaluation_qty > 0).sum() * ORDERING_COST_PER_PRODUCT),
        float(evaluation_qty.sum() / MAX_TOTAL_CAPACITY * 100) if MAX_TOTAL_CAPACITY else np.nan,
    ]
})
display(evaluation)

## 12. Visualize demand vs. optimized quantity

In [ ]:
plot_df = planning.copy()
plot_df["Plot_Label"] = (
    plot_df["Product_Code"].astype(str) + " | " + plot_df["Warehouse"].astype(str)
)
plot_df = plot_df.nlargest(TOP_N_PRODUCTS_TO_PLOT, "Required_Demand").sort_values("Required_Demand")

fig, ax = plt.subplots(figsize=(13, max(6, len(plot_df) * 0.35)))
y = np.arange(len(plot_df))
ax.barh(y - 0.18, plot_df["Required_Demand"], height=0.35, label="Required demand")

if "IP_Order_Quantity" in plot_df.columns:
    opt_col = "IP_Order_Quantity"
elif "LP_Order_Quantity" in plot_df.columns:
    opt_col = "LP_Order_Quantity"
else:
    opt_col = None

if opt_col:
    ax.barh(y + 0.18, plot_df[opt_col], height=0.35, label="Optimized quantity")

ax.set_yticks(y)
ax.set_yticklabels(plot_df["Plot_Label"])
ax.set_xlabel("Units")
ax.set_title("Required Demand vs. Optimized Order Quantity")
ax.legend()
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 13. Export results

In [ ]:
output_path = Path("prescriptive_analytics_optimization_results.csv")
planning.to_csv(output_path, index=False)
print(f"Saved results to: {output_path.resolve()}")

## Interpretation and limitations

- The notebook demonstrates LP, IP/MIP, demand sensitivity, cost evaluation, and visualization.
- Costs and capacity are illustrative inputs and should be replaced with operational values.
- The original dataset does not provide current inventory, supplier lead time, service-level targets, order minimums, or actual cost/revenue data.
- Aggregated historical demand is not automatically a future demand forecast. Define a planning horizon and use a forecast before operational use.
- If total required demand exceeds capacity, the model correctly reports infeasibility rather than claiming an optimal solution.